In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    _p = _root / "paths.py"
    if _p.exists() and "GPT4O_ROOT" in _p.read_text(encoding="utf-8", errors="ignore"):
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find GPT4o/paths.py — set Jupyter cwd to After_PT_Removal/GPT4o or a subfolder."
    )
import paths


In [1]:
import openai
import pandas as pd

# Load the CSV file
df = pd.read_csv(paths.REMOVE_LOW_IRR_DATA / "Qwen72B_annotated_MedPAIR_relevancy.csv")

# Display the first few rows after removing duplicates
print(df)
# df.columns
# print(len(df))

      Origin data_source_df3  \
0     ID0002            jama   
1     ID0003        medxpert   
2     ID0007      medbullets   
3     ID0009            jama   
4     ID0010        medxpert   
...      ...             ...   
1295  ID1995            mmlu   
1296  ID1996            mmlu   
1297  ID1997        medxpert   
1298  ID1998      medbullets   
1299  ID1999        medxpert   

                                        Patient_Profile  \
0     1. A woman in her 60s with a history of hyperl...   
1     1. A 20-year-old woman comes to the primary ca...   
2     1. A 72-year-old man presents to his primary c...   
3     1. A woman in her 30s presented with multiple ...   
4     1. A 17-year-old high school student accidenta...   
...                                                 ...   
1295  1. A 17-year-old girl is brought to the emerge...   
1296  1. A 68-year-old female presents to the emerge...   
1297  1. A 27-year-old woman presents with a 4-month...   
1298  1. A 6-month-old gi

In [2]:
import openai

def generate_direct_prediction(context, question):
    """
    Queries GPT-5 with a clinical vignette (context) and a multiple-choice question (with embedded options).
    Returns only the predicted answer in the format: '[Letter]: [Answer Text]' (e.g., 'B: Femoral artery murmur').
    """
    prompt = f"""
You are given some context and a multiple-choice question.

Select the most appropriate answer from the options provided.

{context}

{question}

Provide your response in the following format:\n<answer>Option [letter]</answer>"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content.strip()

    except Exception:
        return "Error"

In [ ]:
import pandas as pd
from tqdm import tqdm
import os

output_path = paths.PREDICTIONS / "[SR]_gpt4o_predictions_on_qwen72b_removed.csv"

# If continuing from a previous batch, load the existing file and get already-completed indices
if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path)
    completed_ids = set(df_existing.index)
    print(f"✅ Loaded existing file with {len(completed_ids)} completed rows.")
else:
    df_existing = pd.DataFrame()
    completed_ids = set()

# Collect new results in a list of dicts
new_rows = []

# Iterate with progress bar
for idx, row in tqdm(df.iterrows(), total=len(df)):
    if idx in completed_ids:
        continue  # skip already processed

    pred = generate_direct_prediction(row["Qwen72B_High_Relevance"], row["question_options_x"])
    
    result_row = row.to_dict()
    result_row["gpt4o_direct_prediction"] = pred
    new_rows.append(result_row)

    # Write out after each row to ensure persistence
    df_batch = pd.DataFrame(new_rows)
    df_combined = pd.concat([df_existing, df_batch], ignore_index=True)
    df_combined.to_csv(output_path, index=False)


100%|███████████████████████████████████████| 1300/1300 [17:15<00:00,  1.26it/s]


In [5]:
import pandas as pd
import re
import numpy as np

# Load the model predictions
output_path = paths.PREDICTIONS / "[SR]_gpt4o_predictions_on_qwen72b_removed.csv"
df = pd.read_csv(output_path)

# Extract the predicted letter from the XML format - UPDATED to handle brackets
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        # Try to match with optional brackets around the letter
        match = re.search(r"<answer>Option\s+\[?([A-J])\]?</answer>", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

df["gpt_letter"] = df["gpt4o_direct_prediction"].apply(extract_letter_from_xml)

# Clean and standardize the ground truth answer
df["answer_letter"] = df["answer_corr"].astype(str).str.strip().str.upper()

# Compare predictions to ground truth
df["gpt_letter_match"] = df.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Convert to binary for std calculation
df["gpt_letter_binary"] = df["gpt_letter_match"].map({"Correct": 1, "Incorrect": 0})

# Compute overall accuracy
correct_count = df["gpt_letter_binary"].sum()
total_count = df["gpt_letter_binary"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")

# Per-data source accuracy and std
for source in df["data_source_corr_trainee"].unique():
    source_df = df[df["data_source_corr_trainee"] == source]
    correct = source_df["gpt_letter_binary"].sum()
    total = source_df["gpt_letter_binary"].notna().sum()
    acc = correct / total if total > 0 else 0
    std = source_df["gpt_letter_binary"].std(ddof=1) if total > 1 else float("nan")
    
    print(f"Data Source: {source}")
    print(f"  Correct Predictions: {correct}")
    print(f"  Total Predictions: {total}")
    print(f"  Accuracy: {acc:.2%}")
    print(f"  Std Dev: {std:.4f}\n")
    

# Save the updated dataframe with the new columns back to CSV
df["gpt_letter"].to_csv(output_path, index=False)
print(f"\nUpdated data saved to '{output_path}'")

Letter-Based Correct Predictions: 885
Total Predictions Compared: 1300
Letter Match Accuracy: 68.08%
Data Source: jama
  Correct Predictions: 421
  Total Predictions: 582
  Accuracy: 72.34%
  Std Dev: 0.4477

Data Source: medxpert
  Correct Predictions: 121
  Total Predictions: 318
  Accuracy: 38.05%
  Std Dev: 0.4863

Data Source: medbullets
  Correct Predictions: 159
  Total Predictions: 207
  Accuracy: 76.81%
  Std Dev: 0.4231

Data Source: mmlu
  Correct Predictions: 184
  Total Predictions: 193
  Accuracy: 95.34%
  Std Dev: 0.2114


Updated data saved to '[SR]_gpt4o_predictions_on_qwen72b_removed.csv'


In [6]:
import numpy as np

# Collect accuracies per data source
accuracies = []

for source in df["data_source_corr_trainee"].unique():
    source_df = df[df["data_source_corr_trainee"] == source]
    correct = (source_df["gpt_letter_match"] == "Correct").sum()
    total = source_df["gpt_letter_match"].notna().sum()
    acc = correct / total if total > 0 else 0
    accuracies.append(acc)

# Calculate standard deviation
accuracy_std = np.std(accuracies, ddof=1)  # use ddof=1 for sample std deviation
print(f"Standard Deviation of Accuracy Across Data Sources: {accuracy_std:.4f}")

Standard Deviation of Accuracy Across Data Sources: 0.2390
